Middleware overview
===================

Copy pageMiddleware provides a way to more tightly control what happens inside the agent. Middleware is useful for the following:

*   Tracking agent behavior with logging, analytics, and debugging.
    
*   Transforming prompts, [tool selection](https://docs.langchain.com/oss/python/langchain/middleware/built-in#llm-tool-selector), and output formatting.
    
*   Adding [retries](https://docs.langchain.com/oss/python/langchain/middleware/built-in#tool-retry), [fallbacks](https://docs.langchain.com/oss/python/langchain/middleware/built-in#model-fallback), and early termination logic.
    
*   Applying [rate limits](https://docs.langchain.com/oss/python/langchain/middleware/built-in#model-call-limit), guardrails, and [PII detection](https://docs.langchain.com/oss/python/langchain/middleware/built-in#pii-detection).

In [5]:
import os
from dotenv import load_dotenv

load_dotenv()

os.environ["GOOGLE_API_KEY"] = os.getenv("GOOGLE_API_KEY")

### Summarization middleware

Automatically summarize conversation history when approaching token limits, preserving recent messages while compressing older context. Summarization is useful for the following:

*   Long-running conversations that exceed context windows.
    
*   Multi-turn dialogues with extensive history.
    
*   Applications where preserving full conversation context matters.

In [6]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages import HumanMessage, SystemMessage


#messagr summarization
agent = create_agent(
    model="google_genai:gemini-3.5-flash-lite",
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            #model: choose the model to summarize
            model="google_genai:gemini-3.5-flash-lite",
            
            #trigger: condition on which the summarization will trigger, like in this when messages length reaches 10
            trigger=("messages", 10),
            
            #keep: how much context you have to save
            keep=("messages", 4)
        ),
        #next middlware
    ]
)

In [13]:
### Run with thread id
config = {"configurable": {"thread_id": "test-1"}}

In [14]:
#Alternative test data
questions= [
    "what is 2+2?",
    "what is 10*5?",
    "what is 34/2?",
    "what is 34-44?",
    "what is 2**2?",
    "what is 34*4?",
]

for q in questions:
    response = agent.invoke({"messages":[HumanMessage(content=q)]}, config)
    print(f"Messages: {response}")
    print(f"Messages: {len(response['messages'])}")

Messages: {'messages': [HumanMessage(content='what is 2+2?', additional_kwargs={}, response_metadata={}, id='ba8c8f0a-9b41-4613-9f1e-38344d4e7b41'), AIMessage(content=[{'type': 'text', 'text': '2 + 2 = 4', 'extras': {'signature': 'Eo4BCosBAWkUfRM5RNk6aqZxVY67nwFk+F5qHTS8cE4jIsrUejPGT6OllRSuF7iiMaQwS3Wj61N2Ey6fUIisfgn6Z/wWfaYY5976xc04tvg5BN/b4B1xEFFuefzgfMqz4iEi9BFEDf9xcMdBRinhsy3dBD2QM+QdKyzBosGc80OL6kbUjkLMmTeVDDdeoadGxQ=='}}], additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--01a1fcf4-4009-7627-b2d2-ee7d6af1df78-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 8, 'output_tokens': 7, 'total_tokens': 15, 'input_token_details': {'cache_read': 0}})]}
Messages: 2
Messages: {'messages': [HumanMessage(content='what is 2+2?', additional_kwargs={}, response_metadata={}, id='ba8c8f0a-9b41-4613-9f1e-38344d4e7b41'), AIMessage(content=[{'type': 'text

### Token size

In [17]:
from langchain_core.messages import HumanMessage
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.tools import tool
from langchain.agents.middleware import SummarizationMiddleware

@tool
def search_hotels(city: str) -> str:
    """Search hotels - return long response to user more tokens."""
    return f"""Hotels in {city}
            1. Grand hotels - 5 star, $350/night, spa, pool, gym
            2. City Inn - 4 star, $180/night, business center
            3. Budget Stay - 3 star, $75/night, free wifi"""
            
agent = create_agent(
    model="google_genai:gemini-3.5-flash-lite",
    tools=[search_hotels],
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model="google_genai:gemini-3.5-flash-lite",
            trigger=("tokens", 550),
            keep=("tokens", 200)
        ),
    ]
)

config = {"configurable": {"thread_id": "test-2"}}
#token count
def count_token(messages):
    total_chars = sum(len(str(m.content)) for m in messages)
    return total_chars // 4     # 4 chars ~ 1 token

In [18]:
# Run Test
cities = ["Paris", "London", "Tokyo", "New York", "Dubai", "Singapore"]

for city in cities:
    response = agent.invoke(
        {"messages": [HumanMessage(content = f"Find hotels in {city}")]},
        config= config
    )
    
    token = count_token(response["messages"])
    print(f"{city}: ~{token} tokens, {len(response['messages'])} messages")
    print(f"{(response['messages'])}")

Paris: ~208 tokens, 4 messages
[HumanMessage(content='Find hotels in Paris', additional_kwargs={}, response_metadata={}, id='499b8970-a48d-4199-bc1c-70f1ccd42437'), AIMessage(content=[], additional_kwargs={'function_call': {'name': 'search_hotels', 'arguments': '{"city": "Paris"}'}, '__gemini_function_call_thought_signatures__': {'call_15029470773770985174': 'Eo4BCosBAWkUfRNF+YrQ2keqmfKxbAbTrrjltEspQRjhdCBLlOPzfGsWx4dSRutWiYDIERaAutIuvHmNRT9Ylue126ijdKezSHe5j+RbH8FSWT1dMA+oMtZ1AUSF3pnl7bbT4xn+eMhBrhtabhgDvK0zYNruLNYA95mFN7j5PbItFqTUYRTbiOc/bYQFuQUOwg=='}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--01a1fd06-e3a7-709d-9c4a-6d5eaa977a61-0', tool_calls=[{'name': 'search_hotels', 'args': {'city': 'Paris'}, 'id': 'call_15029470773770985174', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 53, 'output_tokens': 16, 'total_tokens': 69, 'input_token_details'